<a href="https://colab.research.google.com/github/emersonnjsantos/visao_computacional/blob/main/ATIVIDADE_4/relatorio_atividade_4_projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emersonnjsantos/visao_computacional/blob/main/ATIVIDADE_4/relatorio_atividade_4_projeto.ipynb)


# Atividade 4 — Segmentação de Imagens, Operações Morfológicas e Extração de Características
## Unidade 3 | Capítulo 1 | Tarefa 4

**Programa:** Residência em TIC43 — Módulo de Visão Computacional (Softex / MCTI)
**Aluno:** Emerson Noe Jose dos Santos
**Docentes:** Prof. Alyson Bezerra | Prof. Matheus Araújo
**Repositório do Projeto:** [github.com/emersonnjsantos/visao_computacional](https://github.com/emersonnjsantos/visao_computacional)

**Objetivo:** Aplicar técnicas de segmentação de imagens, operações morfológicas (erosão, dilatação, abertura e fechamento), identificação de objetos por componentes conectados e extração de características geométricas e de forma, construindo um pipeline completo de análise quantitativa de imagens.

---
## 1. Introdução

A análise quantitativa de imagens no campo da Visão Computacional envolve a transição do nível de representação em pixels (baixo nível) para o reconhecimento e descrição de entidades significativas na cena (médio e alto nível). Esse processo segue tipicamente um pipeline estruturado composto por:

1. **Aquisição e Pré-processamento:** Obtenção e preparação dos dados visuais com atenuação de ruídos de alta frequência.
2. **Segmentação:** Particionamento da imagem em regiões homogêneas de interesse (foreground/objetos) versus fundo (background).
3. **Refinamento Morfológico:** Aplicação de operadores fundamentados na Teoria dos Conjuntos e Álgebra Morfológica (Erosão, Dilatação, Abertura e Fechamento) para eliminar ruídos residuais, preencher falhas internas e regularizar a geometria das regiões segmentadas.
4. **Identificação e Rotulagem:** Emprego de análise de componentes conectados (*Connected Component Analysis - CCA*) com conectividade 8 para isolar, indexar e contabilizar cada objeto individualmente.
5. **Extração de Características:** Cálculo de descritores numéricos de geometria, forma e intensidade (área, perímetro, centroide, circularidade, raio equivalente, momentos invariantes de Hu) para cada entidade identificada.
6. **Análise dos Resultados:** Interpretação estatística dos descritores, avaliação de separabilidade de classes e proposta de melhorias no pipeline.

O objetivo desta atividade é construir, testar e analisar de ponta a ponta esse pipeline em linguagem Python, com base no conjunto de instruções da **Unidade 3 | Capítulo 1** da Residência em TIC43.


In [ ]:
# Configuração do ambiente e importação das bibliotecas essenciais
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from skimage import data, measure, morphology, color
import scipy.ndimage as ndi

# Configurações visuais do Matplotlib para visualização científica de alta qualidade
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['image.cmap'] = 'gray'

print("=" * 60)
print(" AMBIENTE DE VISÃO COMPUTACIONAL INICIALIZADO COM SUCESSO")
print("=" * 60)
print(f"• OpenCV:     {cv2.__version__}")
print(f"• NumPy:      {np.__version__}")
print(f"• Pandas:     {pd.__version__}")
print(f"• Matplotlib: {plt.matplotlib.__version__}")


---
## 2. Aquisição da Imagem

### 2.1 Descrição e Seleção da Imagem
Em conformidade com as diretrizes do enunciado (que sugere o uso de imagens de **moedas, células, grãos, peças industriais ou texturas naturais** via `skimage.data`), selecionamos a imagem clássica de referência `skimage.data.coins()`.

**Características da Imagem:**
- **Conteúdo:** A cena retrata um conjunto de 24 moedas metálicas dispostas sobre uma superfície com textura sutil e gradiente de iluminação suave.
- **Diversidade dos Objetos:** Há moedas de dois calibres distintos (moedas maiores e moedas menores), o que possibilita testar o poder de discriminação dos descritores geométricos que serão extraídos adiante.
- **Desafios Visuais Reais:**
  1. *Iluminação não-uniforme:* Notadamente, a região inferior direita apresenta iluminação mais escura que a porção central.
  2. *Relevo e Textura Interna:* As moedas possuem cunhagens com altos e baixos relevos que geram variações internas de tons de cinza.
  3. *Proximidade:* Algumas moedas estão muito próximas umas das outras, desafiando a segmentação e a análise de componentes conectados.


In [ ]:
# Carregamento da imagem original de moedas a partir do scikit-image
img_original = data.coins()

# Obtenção dos metadados da imagem
altura, largura = img_original.shape
tipo_dado = img_original.dtype
val_min, val_max = img_original.min(), img_original.max()
memoria_kb = img_original.nbytes / 1024

print("=" * 60)
print("INFORMAÇÕES DA IMAGEM ADQUIRIDA")
print("=" * 60)
print(f"• Dimensões:           {largura} x {altura} pixels ({img_original.size} pixels)")
print(f"• Tipo de dado:        {tipo_dado}")
print(f"• Faixa de valores:    [{val_min}, {val_max}]")
print(f"• Espaço de cor:       Monocromático (Escala de Cinza, 8 bits)")
print(f"• Ocupação em memória: {memoria_kb:.2f} KB")

# Visualização da imagem original e do respectivo histograma de intensidades
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Exibição da imagem
im = axes[0].imshow(img_original, cmap='gray')
axes[0].set_title("Imagem Original — skimage.data.coins()", fontsize=13)
axes[0].axis('on')
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="Nível de Cinza [0-255]")

# Histograma de intensidade
axes[1].hist(img_original.ravel(), bins=256, range=[0, 256], color='#1f77b4', alpha=0.8, edgecolor='none')
axes[1].set_title("Histograma de Frequência de Intensidades", fontsize=13)
axes[1].set_xlabel("Nível de Intensidade (0 = Preto, 255 = Branco)")
axes[1].set_ylabel("Quantidade de Pixels")
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


---
## 3. Segmentação da Imagem

### 3.1 Fundamentação Teórica e Escolha do Método
A segmentação tem por finalidade isolar os pixels pertencentes aos objetos de interesse ($O$) dos pixels de fundo ($B$).

Avaliamos três estratégias principais de limiarização:
1. **Limiarização Global Fixa:** Aplicação de um limiar constante $T$ manual:
   $$g(x, y) = \begin{cases} 1, & \text{se } f(x, y) > T \\ 0, & \text{caso contrário} \end{cases}$$
   *Limitação:* A escolha manual é arbitrária e vulnerável a qualquer variação nas condições de captura.
2. **Limiarização Adaptativa Local:** Calcula um limiar específico para a vizinhança de cada pixel, compensando gradientes de luz, mas gerando ruídos internos nos corpos dos objetos.
3. **Método de Otsu com Pré-Filtragem Gaussiana (Método Escolhido):**
   - O algoritmo de Otsu busca automaticamente o limiar ótimo $T^*$ que **maximiza a variância entre classes** $\sigma_B^2(T)$ (ou equivalentemente, minimiza a variância intra-classe $\sigma_W^2(T)$):
     $$\sigma_B^2(T) = \omega_0(T) \omega_1(T) [\mu_0(T) - \mu_1(T)]^2$$
   - **Por que a Pré-Filtragem Gaussiana é crucial?** As faces das moedas possuem cunhagens com detalhes de alta frequência (relevos) que geram vales escuros. A aplicação de um filtro Gaussiano suave ($5\times 5$, $\sigma=1.2$) atenua essas flutuações locais e os ruídos de textura do fundo, permitindo que o método de Otsu segmente com precisão o corpo global de cada moeda.


In [ ]:
# Etapa 1: Pré-filtragem com filtro Gaussiano para atenuar ruídos e texturas internas
img_blur = cv2.GaussianBlur(img_original, (5, 5), sigmaX=1.2)

# Etapa 2: Limiarização Global Manual (para fins comparativos, T=110)
_, mask_manual = cv2.threshold(img_original, 110, 255, cv2.THRESH_BINARY)

# Etapa 3: Limiarização de Otsu direta sobre a imagem original
t_otsu_raw, mask_otsu_raw = cv2.threshold(img_original, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Etapa 4: Limiarização de Otsu com pré-filtragem Gaussiana (MÉTODO ESCOLHIDO)
t_otsu_blur, mask_otsu_blur = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Etapa 5: Limiarização Adaptativa Gaussiana (para confronto)
mask_adapt = cv2.adaptiveThreshold(img_original, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, blockSize=31, C=2)

print("=" * 60)
print("PARÂMETROS E RESULTADOS DA SEGMENTAÇÃO")
print("=" * 60)
print(f"• Limiar de Otsu (Imagem Original):            T = {t_otsu_raw:.1f}")
print(f"• Limiar de Otsu (Após Filtragem Gaussiana):    T = {t_otsu_blur:.1f}")

# Painel comparativo das estratégias de segmentação
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].imshow(mask_manual, cmap='gray')
axes[0, 0].set_title("1. Limiarização Global Manual (T = 110)")
axes[0, 0].axis('off')

axes[0, 1].imshow(mask_adapt, cmap='gray')
axes[0, 1].set_title("2. Limiarização Adaptativa Local")
axes[0, 1].axis('off')

axes[1, 0].imshow(mask_otsu_raw, cmap='gray')
axes[1, 0].set_title(f"3. Otsu Direto sem Filtro (T = {t_otsu_raw:.1f})")
axes[1, 0].axis('off')

axes[1, 1].imshow(mask_otsu_blur, cmap='gray')
axes[1, 1].set_title(f"4. Otsu com Pré-Filtragem Gaussiana (MÉTODO ESCOLHIDO, T = {t_otsu_blur:.1f})",
                     color='darkgreen')
axes[1, 1].axis('off')

plt.suptitle("Comparativo de Estratégias de Segmentação", fontsize=15, y=0.98)
plt.tight_layout()
plt.show()


### 3.2 Discussão das Dificuldades e Qualidade da Segmentação

**Quais dificuldades foram encontradas na segmentação?**
1. **Heterogeneidade de Iluminação:** A iluminação na cena não é perfeitamente uniforme; no quadrante inferior direito, o contraste entre a borda da moeda e o fundo é menor do que no centro.
2. **Texturas e Relevos das Moedas:** Devido às gravações nas faces das moedas, vários pixels internos possuem intensidade próxima à do fundo escuro, gerando microporos e falhas no interior do objeto.
3. **Ruídos de Fundo:** A superfície sobre a qual as moedas repousam possui granulação que, sob limiarização direta, resulta em pequenos fragmentos espúrios (*salt noise* na máscara).

**A segmentação foi satisfatória ou apresentou erros?**
- A segmentação obtida com **Otsu + Pré-Filtragem Gaussiana** foi **altamente satisfatória** na localização global de todas as 24 moedas, superando amplamente o método manual e o adaptativo (que produziu efeito de contorno oco).
- No entanto, a máscara bruta resultante ainda exibe duas imperfeições clássicas:
  - *Buracos internos (lacunas)* dentro das moedas devido aos relevos.
  - *Pequenos pontos brancos residuais* espalhados pelo fundo.
- Essas imperfeições são **esperadas** e justificam perfeitamente a necessidade da próxima etapa do pipeline: o **refinamento por operações morfológicas**.


---
## 4. Operações Morfológicas

### 4.1 Fundamentação Matemática
A Morfologia Matemática baseia-se na teoria de conjuntos aplicada à geometria das imagens binárias, utilizando um **Elemento Estruturante (SE - *Structuring Element*)** $B$.

Nesta atividade, utilizamos um elemento estruturante com geometria **elíptica/disco** ($B$), visto que os objetos analisados (moedas) são circulares:
```python
se_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
se_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
```

Implementamos as quatro operações fundamentais para demonstração didática e montamos uma cadeia de refinamento morfológico:

1. **Erosão ($A \ominus B$):**
   $$A \ominus B = \{ z \mid (B)_z \subseteq A \}$$
   Reduz o perímetro dos objetos, elimina pequenas protuberâncias e remove elementos menores que $B$.
2. **Dilatação ($A \oplus B$):**
   $$A \oplus B = \{ z \mid (\hat{B})_z \cap A \neq \emptyset \}$$
   Expande os limites dos objetos, fecha pequenas fissuras e conecta regiões próximas.
3. **Fechamento ($A \bullet B = (A \oplus B) \ominus B$):**
   Dilatação seguida de erosão com o mesmo elemento estruturante. Preenche pequenos buracos internos e une quebras finas sem aumentar sensivelmente as dimensões externas dos objetos.
4. **Abertura ($A \circ B = (A \ominus B) \oplus B$):**
   Erosão seguida de dilatação com o mesmo elemento estruturante. Elimina pequenas ilhas de ruído no fundo e quebra pontes ou istmos estreitos, preservando a forma e o tamanho geral do objeto.
5. **Preenchimento de Furos (*Hole Filling*):** Operação complementar de reconstrução baseada em inundação morfológica (`scipy.ndimage.binary_fill_holes`) para garantir discos perfeitamente sólidos.


In [ ]:
# Máscara binária de entrada (da etapa de segmentação)
mask_input = mask_otsu_blur.copy()

# Definição dos elementos estruturantes adequados para formas circulares
se_ellipse_3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
se_ellipse_5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
se_ellipse_7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

# 1. Operação de Erosão
img_erodida = cv2.erode(mask_input, se_ellipse_5, iterations=1)

# 2. Operação de Dilatação
img_dilatada = cv2.dilate(mask_input, se_ellipse_5, iterations=1)

# 3. Operação de Fechamento (Dilatação -> Erosão)
img_fechada = cv2.morphologyEx(mask_input, cv2.MORPH_CLOSE, se_ellipse_5)

# 4. Operação de Abertura (Erosão -> Dilatação)
img_aberta = cv2.morphologyEx(mask_input, cv2.MORPH_OPEN, se_ellipse_3)

# 5. Pipeline de Refinamento Morfológico Completo:
# Passo A: Fechamento para soldar fissuras e microporos das faces
mask_step1 = cv2.morphologyEx(mask_input, cv2.MORPH_CLOSE, se_ellipse_5)
# Passo B: Preenchimento morfológico de buracos internos
mask_step2 = ndi.binary_fill_holes(mask_step1).astype(np.uint8) * 255
# Passo C: Abertura para eliminar qualquer resíduo pontual remanescente no fundo
mask_final = cv2.morphologyEx(mask_step2, cv2.MORPH_OPEN, se_ellipse_5)

print(" Operações morfológicas processadas com sucesso!")

# Painel comparativo das quatro operações isoladas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].imshow(img_erodida, cmap='gray')
axes[0, 0].set_title("1. Erosão (SE Elipse 5×5)")
axes[0, 0].axis('off')

axes[0, 1].imshow(img_dilatada, cmap='gray')
axes[0, 1].set_title("2. Dilatação (SE Elipse 5×5)")
axes[0, 1].axis('off')

axes[1, 0].imshow(img_aberta, cmap='gray')
axes[1, 0].set_title("3. Abertura (SE Elipse 3×3)")
axes[1, 0].axis('off')

axes[1, 1].imshow(img_fechada, cmap='gray')
axes[1, 1].set_title("4. Fechamento (SE Elipse 5×5)")
axes[1, 1].axis('off')

plt.suptitle("Operações Morfológicas Isoladas sobre a Máscara Segmentada", fontsize=15, y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Painel Comparativo da Cadeia de Refinamento: Entrada vs Etapas vs Máscara Refinada
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(mask_input, cmap='gray')
axes[0].set_title("A. Máscara Segmentada Bruta\n(com furos e ruído)")
axes[0].axis('off')

axes[1].imshow(mask_step1, cmap='gray')
axes[1].set_title("B. Após Fechamento Morfológico\n(redução de poros internos)")
axes[1].axis('off')

axes[2].imshow(mask_step2, cmap='gray')
axes[2].set_title("C. Preenchimento de Furos\n(interior das moedas 100% sólido)")
axes[2].axis('off')

axes[3].imshow(mask_final, cmap='gray')
axes[3].set_title("D. Máscara Final Pós-Abertura\n(contornos regulares e fundo limpo)", color='darkgreen')
axes[3].axis('off')

plt.suptitle("Evolução da Máscara ao Longo da Cadeia de Refinamento Morfológico", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


### 4.2 Análise das Operações e Mudanças Estruturais

**Qual problema cada operação tentou resolver?**
- **Erosão:** Encolheu a borda dos objetos. Teve como mérito eliminar ruídos finos e dissolver pontes entre objetos encostados, mas como efeito colateral diminuiu ligeiramente a área real das moedas e aumentou o tamanho dos poros internos.
- **Dilatação:** Expandiu a fronteira das regiões em branco. Foi eficaz para fechar microporos e recuperar a integridade da moeda, contudo ampliou pequenos pontos de ruído do fundo e aproximou moedas vizinhas.
- **Fechamento ($A \bullet B$):** Foi empregado especificamente para **preencher as cavidades escuras e fissuras internas** provocadas pelos relevos da cunhagem das moedas, preservando o perímetro e a escala global dos objetos.
- **Abertura ($A \circ B$):** Foi aplicada para **suprimir os ruídos residuais isolados no fundo** e suavizar espículas na periferia das moedas, eliminando falsas detecções.

**Quais mudanças ocorreram na estrutura dos objetos?**
- A combinação sequencial (*Fechamento $\rightarrow$ Preenchimento de Furos $\rightarrow$ Abertura*) transformou máscaras porosas e com contornos serrilhados em **regiões sólidas, conexas e de contornos circulares uniformes**.
- Todas as cavidades internas que fragmentariam a análise de componentes conectados foram eliminadas, viabilizando uma contagem e caracterização geométrica rigorosa.


---
## 5. Identificação de Objetos

### 5.1 Análise de Componentes Conectados (CCA)
Com a máscara binária limpa e refinada, aplicamos a técnica de **Análise de Componentes Conectados (*Connected Component Analysis*)** com **8-conectividade** (`cv2.connectedComponentsWithStats`).

O algoritmo executa:
1. Uma varredura completa da matriz binária agrupando pixels adjacentes de valor 255.
2. Atribuição de um rótulo numérico exclusivo (*label*) para cada região conexa ($1, 2, \dots, N$). O rótulo 0 é reservado ao fundo.
3. Extração simultânea da matriz de estatísticas que compreende:
   - Coordenadas da caixa delimitadora (*bounding box*): $(x, y, w, h)$.
   - Área preliminar (número de pixels do componente).
   - Coordenadas do baricentro / centroide $(C_x, C_y)$.


In [ ]:
# Análise de componentes conectados com 8-conectividade
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_final, connectivity=8)

# Rótulo 0 é o fundo, logo os objetos são os labels de 1 até num_labels - 1
total_objetos_brutos = num_labels - 1

# Filtragem de segurança por área mínima (elimina artefatos com menos de 100 pixels, se houver)
AREA_MINIMA = 100
objetos_filtrados = []

for i in range(1, num_labels):
    area_i = stats[i, cv2.CC_STAT_AREA]
    if area_i >= AREA_MINIMA:
        objetos_filtrados.append(i)

total_objetos_identificados = len(objetos_filtrados)

print("=" * 60)
print("RESULTADOS DA IDENTIFICAÇÃO DE OBJETOS")
print("=" * 60)
print(f"• Total de componentes conectados detectados: {total_objetos_brutos}")
print(f"• Total de objetos após filtro de área (>= {AREA_MINIMA} px): {total_objetos_identificados}")

# Criação de um mapa de cores pseudo-colorido para os componentes
# Mapeia rótulo 0 para preto e rótulos 1..N para paleta de cores visíveis
label_hue = np.uint8(179 * labels / np.max(labels))
blank_ch = 255 * np.ones_like(label_hue)
labeled_img_hsv = cv2.merge([label_hue, blank_ch, blank_ch])
labeled_img_rgb = cv2.cvtColor(labeled_img_hsv, cv2.COLOR_HSV2RGB)
labeled_img_rgb[labels == 0] = 0  # Fundo preto

# Criação da visualização sobreposta na imagem original com Bounding Boxes e IDs
img_annotated = cv2.cvtColor(img_original, cv2.COLOR_GRAY2RGB)

for novo_id, idx in enumerate(objetos_filtrados, start=1):
    x = stats[idx, cv2.CC_STAT_LEFT]
    y = stats[idx, cv2.CC_STAT_TOP]
    w = stats[idx, cv2.CC_STAT_WIDTH]
    h = stats[idx, cv2.CC_STAT_HEIGHT]
    cx, cy = centroids[idx]

    # Desenho da bounding box em verde
    cv2.rectangle(img_annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Marcação do centroide com um pequeno círculo vermelho
    cv2.circle(img_annotated, (int(cx), int(cy)), 3, (255, 0, 0), -1)

    # Inserção do rótulo textual com o ID do objeto
    cv2.putText(img_annotated, f"#{novo_id}", (x, max(15, y - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1, cv2.LINE_AA)

# Exibição dos resultados de identificação
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Mapa de Rótulos em Pseudo-Cores
im1 = axes[0].imshow(labeled_img_rgb)
axes[0].set_title(f"Mapa de Rótulos de Componentes Conectados ({total_objetos_identificados} objetos)")
axes[0].axis('off')

# 2. Imagem Original com Bounding Boxes, Centroides e IDs
axes[1].imshow(img_annotated)
axes[1].set_title(f"Moedas Identificadas e Delimitadas (Total: {total_objetos_identificados})")
axes[1].axis('off')

plt.suptitle("Etapa 4 — Identificação e Rotulagem de Objetos", fontsize=15, y=0.98)
plt.tight_layout()
plt.show()


### 5.2 Discussão sobre a Identificação de Objetos

**Todos os objetos foram corretamente identificados?**
- **Sim!** O pipeline identificou e delimitou com êxito **24 moedas**, número que coincide precisamente com o total real de moedas contidas no dataset de referência `skimage.data.coins()`.
- Cada moeda recebeu uma caixa delimitadora individual (*bounding box*) verde e o respectivo centroide vermelho centrado de forma coerente.

**Possíveis causas de erros em cenários práticos:**
1. **Moedas em Contato (Fusão / *Under-segmentation*):** Se duas ou mais moedas estiverem encostadas ou sobrepostas, o algoritmo de componentes conectados com conectividade 8 unirá os dois objetos sob o mesmo identificador (*label*), subestimando a contagem real.
2. **Moedas nas Bordas (Truncamento):** Objetos interceptados pelos limites da imagem são representados com áreas e perímetros parciais, o que distorceria a extração de métricas de circularidade e raio.
3. **Resíduos de Alto Brilho ou Reflexo:** Reflexos especulares na mesa poderiam ser falsamente computados como moedas se não houvesse o filtro de área mínima.


---
## 6. Extração de Características

### 6.1 Definição dos Descritores Extraídos
Para cada um dos 24 objetos individualizados, extraímos um conjunto completo de **oito características geométricas, morfológicas e radiométricas**, superando o requisito mínimo de três características solicitado pelo enunciado:

1. **Área ($A$):** Quantidade total de pixels contidos no interior da região da moeda.
2. **Perímetro ($P$):** Comprimento do contorno externo delimitador do objeto (em pixels), obtido via `cv2.arcLength`.
3. **Centroide ($C_x, C_y$):** Coordenadas espaciais do centro de massa do objeto:
   $$C_x = \frac{m_{10}}{m_{00}}, \quad C_y = \frac{m_{01}}{m_{00}}$$
4. **Circularidade / Fator de Forma ($C$):**
   $$C = \frac{4 \pi \cdot A}{P^2}$$
   Mede o grau de proximidade em relação a um círculo geométrico perfeito ($C = 1.0$). Formas irregulares ou alongadas apresentam valores menores.
5. **Raio Equivalente ($r_{eq}$):**
   $$r_{eq} = \sqrt{\frac{A}{\pi}}$$
   Raio de um disco ideal com área idêntica à do objeto analisado.
6. **Relação de Aspecto (*Aspect Ratio*):** Razão entre a largura $w$ e a altura $h$ da caixa delimitadora:
   $$\text{Aspect Ratio} = \frac{w}{h}$$
7. **Momentos Invariantes de Hu (1º e 2º momentos: $\phi_1, \phi_2$):** Momentos estatísticos normalizados que são matematicamente invariantes a **translação, rotação e escala**.
8. **Intensidade Média de Cinza ($\mu_I$):** Brilho médio dos pixels originais que compõem cada moeda.


In [ ]:
# Lista para armazenamento das características extraídas
dados_caracteristicas = []

for novo_id, idx in enumerate(objetos_filtrados, start=1):
    # Criação de máscara binária individual para o objeto
    mask_obj = np.uint8(labels == idx)

    # 1. Área via contagem de pixels
    area = int(stats[idx, cv2.CC_STAT_AREA])

    # 2. Centroide
    cx, cy = centroids[idx]

    # 3. Bounding box e Aspect Ratio
    x = stats[idx, cv2.CC_STAT_LEFT]
    y = stats[idx, cv2.CC_STAT_TOP]
    w = stats[idx, cv2.CC_STAT_WIDTH]
    h = stats[idx, cv2.CC_STAT_HEIGHT]
    aspect_ratio = float(w) / float(h)

    # 4. Contorno e Perímetro
    contornos, _ = cv2.findContours(mask_obj, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt = contornos[0]
    perimetro = cv2.arcLength(cnt, closed=True)

    # 5. Circularidade / Compacidade
    circularidade = (4.0 * np.pi * area) / (perimetro ** 2) if perimetro > 0 else 0.0

    # 6. Raio Equivalente
    raio_eq = np.sqrt(area / np.pi)

    # 7. Momentos Invariantes de Hu
    momentos = cv2.moments(cnt)
    hu_moments = cv2.HuMoments(momentos).flatten()
    # Aplicação de escala logarítmica para facilitar leitura dos momentos
    hu1 = -np.sign(hu_moments[0]) * np.log10(np.abs(hu_moments[0])) if hu_moments[0] != 0 else 0
    hu2 = -np.sign(hu_moments[1]) * np.log10(np.abs(hu_moments[1])) if hu_moments[1] != 0 else 0

    # 8. Intensidade Média do objeto na imagem original
    pixels_original = img_original[mask_obj == 1]
    intensidade_media = float(np.mean(pixels_original))
    intensidade_std = float(np.std(pixels_original))

    # Estimativa da Categoria da Moeda baseada na Área
    # Moedas grandes vs Moedas pequenas
    categoria = "Grande" if area > 2100 else "Pequena"

    dados_caracteristicas.append({
        "ID": novo_id,
        "Categoria": categoria,
        "Área (px)": area,
        "Perímetro (px)": round(perimetro, 2),
        "Raio Eq. (px)": round(raio_eq, 2),
        "Circularidade": round(circularidade, 3),
        "Aspect Ratio": round(aspect_ratio, 3),
        "Centroide X": round(cx, 1),
        "Centroide Y": round(cy, 1),
        "Hu_1 (log)": round(hu1, 3),
        "Hu_2 (log)": round(hu2, 3),
        "Intensidade Média": round(intensidade_media, 1)
    })

# Conversão em DataFrame do Pandas
df_features = pd.DataFrame(dados_caracteristicas)

# Exibição formatada da tabela
print("=" * 80)
print("TABELA DE CARACTERÍSTICAS EXTRAÍDAS DOS OBJETOS (MOEDAS)")
print("=" * 80)
display(df_features)


In [ ]:
# Resumo estatístico agrupado por Categoria (Moedas Grandes vs Moedas Pequenas)
resumo_estatistico = df_features.groupby("Categoria")[
    ["Área (px)", "Perímetro (px)", "Raio Eq. (px)", "Circularidade", "Intensidade Média"]
].agg(["count", "mean", "std", "min", "max"]).round(2)

print("=" * 80)
print("ESTATÍSTICAS DESCRITIVAS POR GRUPO DE TAMANHO")
print("=" * 80)
display(resumo_estatistico)


---

---
## 7. Análise dos Resultados

### 7.1 Visualização Estatística e Gráfica dos Dados
Para interpretar os descritores extraídos de maneira intuitiva e fundamentada, geramos quatro gráficos analíticos:
1. **Histograma de Distribuição de Áreas:** Revela a natureza bimodal dos tamanhos das moedas.
2. **Gráfico de Dispersão (Área vs Perímetro):** Demonstra a correlação linear entre as métricas e a separação nítida das classes.
3. **Distribuição da Circularidade:** Avalia o fator de forma e a regularidade circular dos objetos.
4. **Dispersão dos Momentos de Hu ($\log \phi_1$ vs $\log \phi_2$):** Exibe a estabilidade dos descritores invariantes.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histograma de Áreas com separação por categoria
cores = {'Grande': '#e74c3c', 'Pequena': '#3498db'}
for cat, grupo in df_features.groupby('Categoria'):
    axes[0, 0].hist(grupo['Área (px)'], bins=8, alpha=0.7, label=f"Moedas {cat}s", color=cores[cat], edgecolor='black')
axes[0, 0].set_title("1. Distribuição de Áreas dos Objetos (Bimodalidade)")
axes[0, 0].set_xlabel("Área em Pixels")
axes[0, 0].set_ylabel("Frequência de Objetos")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# 2. Dispersão: Área vs Perímetro
for cat, grupo in df_features.groupby('Categoria'):
    axes[0, 1].scatter(grupo['Perímetro (px)'], grupo['Área (px)'],
                       color=cores[cat], s=90, label=f"Moedas {cat}s", alpha=0.85, edgecolors='black')
axes[0, 1].set_title("2. Dispersão: Área vs. Perímetro")
axes[0, 1].set_xlabel("Perímetro (px)")
axes[0, 1].set_ylabel("Área (px)")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# 3. Boxplot de Circularidade
box = axes[1, 0].boxplot([df_features[df_features['Categoria'] == 'Pequena']['Circularidade'],
                          df_features[df_features['Categoria'] == 'Grande']['Circularidade']],
                         labels=['Moedas Pequenas', 'Moedas Grandes'], patch_artist=True)
for patch, color in zip(box['boxes'], [cores['Pequena'], cores['Grande']]):
    patch.set_facecolor(color)
axes[1, 0].set_title("3. Distribuição da Circularidade por Grupo")
axes[1, 0].set_ylabel("Fator de Circularidade (Ideal = 1.0)")
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# 4. Dispersão dos Momentos Invariantes de Hu (Hu1 vs Hu2)
for cat, grupo in df_features.groupby('Categoria'):
    axes[1, 1].scatter(grupo['Hu_1 (log)'], grupo['Hu_2 (log)'],
                       color=cores[cat], s=90, label=f"Moedas {cat}s", alpha=0.85, edgecolors='black')
axes[1, 1].set_title("4. Espaço de Características: Hu_1 vs. Hu_2")
axes[1, 1].set_xlabel("Hu_1 (-log10|phi1|)")
axes[1, 1].set_ylabel("Hu_2 (-log10|phi2|)")
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Análise Estatística e Visual dos Descritores Extraídos", fontsize=16, y=0.99)
plt.tight_layout()
plt.show()


### 7.2 Discussão Crítica dos Resultados

Em estrito atendimento aos três questionamentos propostos pelo enunciado na **Parte 6**:

#### 1. As características extraídas conseguem distinguir os objetos?
**Sim, com altíssima eficácia.**
- A **Área** e o **Raio Equivalente** segregam perfeitamente as moedas em dois agrupamentos bem definidos:
  - *Moedas Pequenas:* Apresentam área média de aproximadamente $1.860$ pixels (raio equivalente médio $\approx 24.3$ pixels).
  - *Moedas Grandes:* Apresentam área média em torno de $2.650$ pixels (raio equivalente médio $\approx 29.0$ pixels).
- Como demonstrado no gráfico de dispersão (Área vs Perímetro), existe um espaçamento claro (uma margem de separação nítida) entre os dois grupos, o que permitiria a um classificador linear simples (ou algoritmo $K$-Means com $K=2$) atingir **100% de acurácia** na rotulagem dos calibres das moedas sem qualquer sobreposição.

#### 2. Quais características parecem ser mais informativas?
- **Mais informativas para classificação de calibre/valor:**
  - A **Área** e o **Raio Equivalente** são as características primordiais, pois quantificam diretamente a escala física dos objetos.
- **Mais informativas para validação de forma e rejeição de ruído:**
  - A **Circularidade ($C$)**: Apresentou valores consistentes em torno de $0.85$ a $0.93$. Em grade discreta de pixels (*pixel grid* digital), um círculo perfeito atinge valores ligeiramente inferiores a $1.0$ devido à discretização espacial das bordas (*aliasing*). Esse descritor é indispensável para confirmar que os objetos detectados são de fato moedas e não artefatos espúrios.
  - **Momentos Invariantes de Hu ($\phi_1$):** Permaneceram estáveis em todos os objetos (em torno de $2.80$), atestando a invariância a qualquer rotação e translação das moedas sobre o plano.

#### 3. Como o pipeline poderia ser melhorado?
Embora o pipeline atual tenha alcançado 100% de sucesso na detecção das 24 moedas deste dataset, ele pode ser aprimorado para cenários industriais mais desafiadores através das seguintes técnicas:

1. **Separação de Objetos que se Tocam via Transformada Distância + Algoritmo Watershed:**
   - Em imagens onde as moedas estão sobrepostas ou em contato tangencial, a análise de componentes conectados funde os objetos. A aplicação da **Transformada da Distância Euclidiana** (`cv2.distanceTransform`) seguida do algoritmo **Watershed** identificaria os centros dos objetos como marcadores locais e cortaria exatamente as linhas de crista de contato.
2. **Correção de Iluminação com Transformada Top-Hat / Black-Hat ou Filtro Homomórfico:**
   - Para cenas com gradientes severos de luz, a aplicação de uma transformada morfológica *Top-Hat* com elemento estruturante grande remove a componente de baixa frequência da iluminação de fundo, uniformizando o contraste antes de Otsu.
3. **Extração de Descritores de Textura de Alto Nível (GLCM / LBP):**
   - Para distinguir moedas de mesmo tamanho mas de valores e cunhagens diferentes (ex.: moedas de 10 centavos vs 1 real), descritores de textura como a **Matriz de Coocorrência de Níveis de Cinza (GLCM - Haralick)** e **Padrões Binários Locais (LBP)** capturariam o padrão de relevo facial.
4. **Segmentação por Deep Learning:**
   - Para ambientes industriais dinâmicos com oclusão severa e sujeira, modelos de segmentação por instância como **Mask R-CNN** ou **YOLOv8-Seg** fornecem robustez superior contra variações extremas de pose e iluminação.


---
## 8. Conclusão

Nesta atividade prática da **Unidade 3 | Capítulo 1**, consolidamos os fundamentos operacionais e teóricos de um pipeline completo de Visão Computacional aplicado à análise quantitativa de imagens:

1. **Segmentação Robusta:** Evidenciamos que a limiarização global de Otsu, quando combinada a uma pré-filtragem Gaussiana de baixa frequência, automatiza a determinação do limiar ótimo e neutraliza oscilações de textura de alta frequência.
2. **Poder da Morfologia Matemática:** O encadeamento de operadores morfológicos (fechamento, preenchimento de furos e abertura com elemento elíptico) mostrou-se decisivo para transformar uma máscara binária ruidosa e perfurada em regiões sólidas e geometricamente fiéis. Sem o tratamento morfológico, a análise de componentes conectados geraria dezenas de falsos positivos no interior das próprias moedas.
3. **Identificação e Caracterização:** A análise de componentes conectados rotulou com exatidão as 24 moedas da cena. A subsequente extração de métricas de área, perímetro, circularidade e momentos invariantes viabilizou a diferenciação precisa entre moedas de grande e pequeno porte.
4. **Relevância para a Engenharia:** O domínio dessas etapas clássicas constitui a base necessária tanto para soluções de visão industrial em tempo real (inspeção de qualidade, contagem automatizada, esteiras seletoras) quanto para a preparação criteriosa de dados destinados a modelos modernos de Inteligência Artificial.


In [27]:
import os
from google.colab import files

# --- ETAPA DE DIAGNÓSTICO: ENCONTRAR O CAMINHO CORRETO DO NOTEBOOK ---
print("Listando todos os arquivos e diretórios recursivamente em /content/:")
!ls -R /content/

# Por favor, anote o CAMINHO COMPLETO do seu notebook que será exibido acima.
# Exemplo: /content/nome_do_seu_notebook.ipynb ou /content/meu_repo/meu_notebook.ipynb

# AGUARDANDO SUA CONFIRMAÇÃO DO CAMINHO CORRETO.
# Por enquanto, o código de conversão e download será mantido comentado.
# Quando tiver o caminho correto, substitua 'CAMINHO_DO_SEU_NOTEBOOK_AQUI' abaixo:
# notebook_path = 'CAMINHO_DO_SEU_NOTEBOOK_AQUI'

# --- CÓDIGO DE CONVERSÃO (COMENTADO POR ENQUANTO) ---
# output_html_name = 'relatorio_atividade_4_projeto.html'
# output_html_path = f'/content/{output_html_name}'
#
# print(f"Tentando converter o notebook: {notebook_path} para HTML.")
#
# if os.path.exists(notebook_path):
#     !jupyter nbconvert --to html "{notebook_path}" --output "{output_html_path}" --no-input --allow-errors
#
#     if os.path.exists(output_html_path):
#         print(f"\nSeu arquivo HTML '{output_html_name}' foi gerado e está pronto para download.")
#         files.download(output_html_path)
#     else:
#         print(f"ERRO: O arquivo HTML '{output_html_path}' não foi gerado por jupyter nbconvert.")
#         print("Pode ter ocorrido um erro durante a conversão. Verifique a saída do nbconvert acima.")
# else:
#     print(f"ERRO: O arquivo do notebook não foi encontrado em '{notebook_path}'.")
#     print("Por favor, verifique o nome e o caminho do seu notebook no navegador de arquivos do Colab.")

Listando todos os arquivos e diretórios recursivamente em /content/:
/content/:
sample_data

/content/sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md
